In [ ]:
from tqdm import tqdm
import pandas as pd
import spacy
import re
import unicodedata
from unidecode import unidecode
import ast
from collections import Counter


In [2]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
ents = pd.read_csv("/Users/pol/Downloads/extracted_ents_ALL.csv", sep=';')
ents = ents.dropna(subset=["name"])
ents

press = pd.read_csv("/Users/pol/Downloads/agg_matches_ALL.csv", sep=';')
press

,article_id,article_text,name,name_count
0,1,Serge Reggiani - Succès & confidences .\n Une ...,"['Serge Reggiani - Succès & confidences', 'Reg...",1
1,2,"Désolés, Warren .\n C'est une maigre consolati...","['Warren', 'Warren Zevon', ""I'll sleep when I'...",10
2,3,Adam Green - Friends of mine .\n Dire qu'on a ...,"['Adam Green - Friends of mine', 'The Kills', ...",1
3,4,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"['gorgée', 'Voz', ""Voix de l'amour"", 'Velocida...",2
4,5,Herman Düne - Mas cambios et Mash Concrete Met...,"['Herman Düne - Mas cambios', 'Mash Concrete M...",1
...,...,...,...,...
52639,52690,"Rap, jazz, rock, classique, chanson… Les album...","['Rap', 'Monde', 'An Evening with Silk Sonic',...",45
52640,52691,Que sait-on vraiment des surrisques de contami...,"['Covid-19', 'Omicron', 'Jean Castex', 'Covid-...",482
52641,52692,"Rencontre avec Riccardo Muti, maestro à vie .\...","['Riccardo Muti', 'Riccardo Muti', 'Cristina',...",70
52642,52693,Ils sont passés chez Colors : la sélection mus...,"['Colors', 'Monde Afrique', 'Le Monde Afrique'...",27


### check texts with possible false positives

In [99]:
count = 0

for i, (text, namelist) in enumerate(
    zip(press["article_text"],
        press["name"]
        )):
    if ("'Johnny Cash'" in namelist and "'Johnny'"):
        #print(i)
        count += 1

print(count)

220


In [44]:
press["name"] = press["name"].apply(ast.literal_eval)
press["name"]

0        [Serge Reggiani - Succès & confidences, Reggia...
1        [Warren, Warren Zevon, I'll sleep when I'm dea...
2        [Adam Green - Friends of mine, The Kills, Gree...
3        [gorgée, Voz, Voix de l'amour, Velocidade, Ces...
4        [Herman Düne - Mas cambios, Mash Concrete Meta...
                               ...                        
52639    [Rap, Monde, An Evening with Silk Sonic, Silk ...
52640    [Covid-19, Omicron, Jean Castex, Covid-19, Clu...
52641    [Riccardo Muti, Riccardo Muti, Cristina, Ricca...
52642    [Colors, Monde Afrique, Le Monde Afrique, Colo...
52643    [Pierre-Emmanuel Barbier, Thomson, Diapason, P...
Name: name, Length: 52644, dtype: object

In [90]:
def count_name_cooc(target_name, position="first"):
    """
    namelists: iterable of lists of extracted names
    target_name: string (e.g. "christophe" or "mae")
    position: "first" or "last"
    """

    target_name = target_name.lower()
    alone_count = 0
    cooccurrence_counter = Counter()

    for namelist in press["name"]:

        names_lower = [n.lower() for n in namelist]

        # Case 1: FIRST NAME logic
        if position == "first":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.startswith(target_name + " ")]

            if has_alone and not full_names:
                alone_count += 1

            elif has_alone and full_names:
                for full_name in full_names:
                    last_part = full_name[len(target_name)+1:]
                    cooccurrence_counter[last_part] += 1

        # Case 2: LAST NAME logic
        elif position == "last":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.endswith(" " + target_name)]

            if has_alone and not full_names:
                alone_count += 1

            elif has_alone and full_names:
                for full_name in full_names:
                    first_part = full_name[:-(len(target_name)+1)]
                    cooccurrence_counter[first_part] += 1

        else:
            raise ValueError("position must be 'first' or 'last'")

    return alone_count, cooccurrence_counter

In [107]:
target_name = "reich"

alone, with_last = count_name_cooc(
    target_name=target_name,
    position="last"
)

print(f"'{target_name}' alone:", alone, "\n")
print(f"'{target_name}' with:", "\n")
for k, v in with_last.most_common():
    print(f"'{k}': {v}")

'reich' alone: 44 

'reich' with: 

'steve': 12
'iiie': 8
'wilhelm': 2
'ive': 2
'maître forestier du': 1
'iie': 1


In [71]:
n = 10                       
count = 0
artist = "Rolling"

for i, (text, namelist) in enumerate(zip(press["article_text"],
                                         press["name"])):
    if artist in namelist:
        print(i)
        print(namelist)
        print("-------------------")
        print(text)
        print("===================")

        count += 1

print(f"{count} OCCURRENCES OF '{artist}'")

1332
['Kanye West', 'Curtis', 'Rolling', 'Kanye West', 'Muhammad Ali', 'Bush', 'Katrina', 'Daft Punk', 'Coldplay', '50 Cent', 'Mike Tyson', 'Sonny Liston', 'Curtis', 'Laurent Rigoulet', 'CD Barclay', 'CD Polydor']
-------------------
Kanye West. Graduation - 50 Cent. Curtis .
 En couverture du Rolling Stone américain, ils se toisent méchamment comme deux boxeurs à la veille du match du siècle. A gauche, 50 Cent, à droiteKanye West, deux poids lourds du rap US dont les albums sont sortis le même jour de septembre, histoire de créer l'événement et de redonner un peu de muscle à l'industrie du disque moribonde. Même s'ils boxent tous deux dans la catégorie multiplatine, ces deux stars de l'Amérique noire s'appuient sur des styles très différents. 
 Kanye West est un genre de Muhammad Ali rap, agile et malin, un baratineur éclairé et engagé (ses fameuses sorties contre Bush après la catas­trophe Katrina), doublé d'un technicien hors pair, dont les arrangements tissent d'habiles variations 

In [69]:
n = 10                       
count = 0
artist = "Rolling"

for i, (text, namelist) in enumerate(zip(press["article_text"],
                                         press["name"])):
    if artist in namelist:
        # split the article into tokens and look for the artist token(s)
        words = text.split()
        for idx, w in enumerate(words):
            if w.lower() == artist.lower():
                lo = max(0, idx - n)
                hi = idx + n + 1
                snippet = " ".join(words[lo:hi])
                print("------------------------")
                print(i)
                print(namelist)
                print(snippet) # only n words before/after
        count += 1

print("==================")
print(f"{count} OCCURRENCES OF '{artist}'")

------------------------
1332
['Kanye West', 'Curtis', 'Rolling', 'Kanye West', 'Muhammad Ali', 'Bush', 'Katrina', 'Daft Punk', 'Coldplay', '50 Cent', 'Mike Tyson', 'Sonny Liston', 'Curtis', 'Laurent Rigoulet', 'CD Barclay', 'CD Polydor']
West. Graduation - 50 Cent. Curtis . En couverture du Rolling Stone américain, ils se toisent méchamment comme deux boxeurs à
------------------------
1446
['Patti Smith', 'Lou Reed', 'Phrase terrible', 'Lou Reed', 'Velvet Underground', 'Robert Christgau', 'Rolling', 'Reed', 'Lou Reed', 'Velvet Undergound', 'Lou Reed', 'Reed', 'Velvet', "I'll be your mirror", 'Pale Blue Eyes', 'Lou Reed', 'Velvet', 'Caroline says II', 'Lou Reed', 'Laurie Anderson', 'Reed', 'tai-chi', 'Cary Grant', 'Lou Reed', '- Reed', 'Magic and loss', 'Julian Schnabel', 'Lola', 'Schnabel', 'Hal Willner', 'Bob Ezrin', 'Steve Hunter', 'Lou Reed', 'Reed', 'Susan Feldman', 'Andy Warhol', 'Songs for Drella', 'John Cale', 'Julian Schnabel', 'Bob Ezrin', 'Steve Hunter', 'Lou Reed', 'Transf

In [76]:
df = ents
len(df)

408677

In [98]:
df = ents

# normalize both texts
def clean_text(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = unidecode(text) # remove accents
    # text = text.replace("\xa0", " ")
    text = text.strip()
    return text


df = df[df["name"].map(len) > 2] # short names

df["name"] = df["name"].apply(clean_text)

### check names with many tokens

maybe leave for now, if we do strict matching they'll be dropped anyway

In [103]:
df = df.drop_duplicates(subset=["name"])
df.reset_index()

,index,name_id,name,name_count,article_id
0,2,3,liberation,3336,870
1,3,4,figaro,2749,1786
2,4,5,internet,2376,40
3,5,6,mozart,1831,87
4,6,7,dieu,1616,592
...,...,...,...,...,...
386005,408671,408672,i'm changing,1,49038
386006,408672,408673,i'm blind,1,48689
386007,408673,408674,i'm amazing,1,49542
386008,408674,408675,i'm all smiles,1,50227


In [105]:
df.to_csv("/Users/pol/Downloads/extracted_ents_clean.csv", sep=';')